# Save A Journey of Faith to Google Drive

This notebook downloads the video sections from GitHub, verifies them, combines them, and saves one MP4 in **My Drive / Abbas Table**. It needs about 1 GB of free Drive space.

1. Press the play button beside the code below. If Google Colab shows a warning, choose **Run anyway**.
2. When prompted, connect the Google account where you want the video saved and allow access to Drive.
3. Leave this page open until you see **Finished**. You do not have to join or upload the sections yourself.

If the video is already present and verified, it will not be downloaded again. An existing file with different contents will be kept.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import os
import shutil
import tempfile
import time
import urllib.request
import uuid

BASE = 'https://raw.githubusercontent.com/wambetebenjamin/Videos/8cf16caa63617a5b5c3f145604f0d3c53327d2fe/A_Journey_of_Faith/chunks/'
CHUNKS = [(94371840, 'fcac82ee5b75ea0b4393d3fed47503eaeea283668e2f617207ced34686332b4a'), (94371840, '25d5a32c7c285d1ad8cc138a2c67166c4246162d517a48da781c74dd7746acea'), (94371840, 'ee6e5557f59c564ffa2b75eb6db7d2df3e02afe1555389eff9f7ffbe99359c0b'), (94371840, '3c87f4649b07cbb1111ee62aa4655b232c6daf456dce363fbd159f22140ab437'), (94371840, '14e4def0288c1c236b46dbc43f5bb8b73c29f502a7af94c6581666945727701e'), (94371840, '8c873e48171de3947a60a9666d8c9ddcb039c8f76bc26470d35f11b4abb66716'), (94371840, '15b4461750c9f3d49928f4e3c52e671747588203d68714f6efba6143e4a851fe'), (94371840, 'f039592a7f7f8231c0aa58ba034a6a55a54607de39d6d69c71787ed04be7d5ae'), (94371840, '00bb65404b40cac95a9181e2b3c8e8731dfafba7f9ebac2626f5a06b5489d6c1'), (94371840, '06278a955276240e2e4614d5bc6cb290513308be3442b237455be89fd6245df4'), (25874506, '9a8e99d172c511700c581eac5fbdc15fd2d19a8036f87d65a75521805efb1c7f')]
TOTAL_HASH = 'e28d64890fc87a878dd94bcbd728976b0d23708515c96da4496032ee67b67b29'


def file_hash(path):
    digest = hashlib.sha256()
    with path.open('rb') as source:
        while block := source.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()


def download_part(url, expected_size, expected_hash, path):
    for attempt in range(1, 4):
        try:
            digest = hashlib.sha256()
            count = 0
            with urllib.request.urlopen(url, timeout=120) as response, path.open('wb') as out:
                while block := response.read(1024 * 1024):
                    count += len(block)
                    if count > expected_size:
                        raise ValueError('Downloaded section is larger than expected')
                    digest.update(block)
                    out.write(block)
            if count != expected_size or digest.hexdigest() != expected_hash:
                raise ValueError('Downloaded section did not pass verification')
            return
        except (OSError, ValueError) as error:
            path.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError('Could not download and verify a section') from error
            print('Trying this section again...', flush=True)
            time.sleep(2 * attempt)


def save_video(folder, base, chunks, total_hash):
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / 'A Journey of Faith.mp4'
    if target.exists():
        print('Checking the video already in your Drive...', flush=True)
        if file_hash(target) == total_hash:
            print('The verified video is already saved at:', target, flush=True)
            return target
        n = 2
        while True:
            target = folder / f'A Journey of Faith ({n}).mp4'
            if not target.exists():
                break
            n += 1
        print('An existing file will be kept. Saving a separate copy.', flush=True)

    with tempfile.TemporaryDirectory(prefix='journey_') as temp:
        temp = Path(temp)
        combined = temp / 'combined.mp4'
        part = temp / 'part.bin'
        whole = hashlib.sha256()
        with combined.open('wb') as output:
            for i, (size, expected) in enumerate(chunks):
                print(f'Downloading section {i + 1} of {len(chunks)}...', flush=True)
                download_part(base + f'part{i:02}.bin', size, expected, part)
                with part.open('rb') as source:
                    while block := source.read(1024 * 1024):
                        whole.update(block)
                        output.write(block)
                part.unlink()
        if whole.hexdigest() != total_hash:
            raise RuntimeError('The combined video did not pass verification')
        print('Download verified. Saving the video to Google Drive...', flush=True)
        partial = folder / ('.upload_' + uuid.uuid4().hex + '.part')
        try:
            copied = 0
            with combined.open('rb') as source, partial.open('xb') as dest:
                while block := source.read(1024 * 1024):
                    dest.write(block)
                    copied += len(block)
                    if copied % (100 * 1024 * 1024) == 0:
                        print(f'Saved {copied // (1024 * 1024)} MB to Drive...', flush=True)
            print('Checking the saved file...', flush=True)
            if file_hash(partial) != total_hash:
                raise RuntimeError('The Drive copy did not pass verification')
            if target.exists():
                raise FileExistsError('The destination appeared during upload. No file was replaced.')
            os.rename(partial, target)
        finally:
            partial.unlink(missing_ok=True)
    print('Finished. Your video is in My Drive, inside the Abbas Table folder:', flush=True)
    print(target.name, flush=True)
    return target

print('Connect the Google account where you want the video saved.', flush=True)
drive.mount('/content/drive')
save_video(Path('/content/drive/MyDrive/Abbas Table'), BASE, CHUNKS, TOTAL_HASH)
